In [ ]:
# Standard library
import math

import geopandas as gpd

# Third-party core libraries
import numpy as np
import pandas as pd

# Geographic calculations
from geographiclib.geodesic import Geodesic
from geopandas import GeoDataFrame
from geopy.distance import distance, geodesic
from pyproj import CRS, Transformer
from shapely import affinity
from shapely.geometry import LineString, Point, Polygon

# Machine learning / spatial indexing
from sklearn.neighbors import BallTree, KDTree


In [2]:
def geodesic_circle_polygons(lat0, lon0, radii_m, n=360, start_bearing=0.0):
    """
    Create geodesic circles (as polygons) around a lat/lon center.

    Parameters
    ----------
    lat0, lon0 : float
        Center latitude and longitude.
    radii_m : list[float]
        Radii of the circles (in meters).
    n : int
        Number of vertices per circle (default 360).
    start_bearing : float
        Bearing offset in degrees (default 0.0).

    Returns
    -------
    GeoDataFrame
        Polygons representing geodesic circles.
    """
    g = Geodesic.WGS84
    bearings = start_bearing + 360 * np.arange(n) / n

    polys = []
    rows = []
    for r in radii_m:
        ring = []
        for brg in bearings:
            out = g.Direct(lat0, lon0, brg, r)
            ring.append((out["lon2"], out["lat2"]))  # (x=lon, y=lat)
        ring.append(ring[0])  # close polygon

        polys.append(Polygon(ring))
        rows.append({"radius_m": r})

    gdf = gpd.GeoDataFrame(rows, geometry=polys, crs="EPSG:4326")
    return gdf

In [3]:
def geodesic_circle_points(lat0, lon0, radii_m, n, start_bearing=0.0):
    """
    Generate geodesic circle points around a latitude/longitude center.

    This function computes points that lie on one or more concentric circles 
    centered at (lat0, lon0). Each circle is defined by a geodesic radius in meters 
    and is approximated using `n` evenly spaced bearings. The output is returned 
    as a GeoDataFrame of points in EPSG:4326 (WGS84).

    Parameters
    ----------
    lat0 : float
        Latitude of the circle center in decimal degrees.
    lon0 : float
        Longitude of the circle center in decimal degrees.
    radii_m : list of float
        Radii of the circles in meters. One set of points will be generated per radius.
    n : int
        Number of points to generate along each circle (higher values give smoother circles).
    start_bearing : float, optional, default=0.0
        Bearing offset in degrees for the first point on each circle (0° = north).

    Returns
    -------
    gdf : GeoDataFrame
        A GeoDataFrame in EPSG:4326 containing the generated points with attributes:
        
        - ``lat`` : latitude of the point
        - ``lon`` : longitude of the point
        - ``bearing_deg`` : bearing from the center to the point
        - ``radius_m`` : radius of the circle in meters
        - ``geometry`` : shapely Point object in EPSG:4326

    Notes
    -----
    - The circle is geodesic, meaning it accounts for the curvature of the Earth.
    - The result consists of **points along the circumference**, not filled polygons.
      If you want a polygon representation, you can construct one by connecting 
      the points per radius.

    Examples
    --------
    >>> gdf = geodesic_circle_points(49.91239, -111.76254, [100, 200], 360)
    >>> gdf.head()
          lat        lon  bearing_deg  radius_m                     geometry
    0  49.9133 -111.76254          0.0       100  POINT (-111.76254 49.9133)
    1  49.9133 -111.76249          1.0       100  POINT (-111.76249 49.9133)
    """
    g = Geodesic.WGS84
    bearings = start_bearing + 360 * np.arange(n) / n

    rows = []
    for r in radii_m:
        for brg in bearings:
            out = g.Direct(lat0, lon0, brg, r)
            rows.append({
                "lat": out["lat2"],
                "lon": out["lon2"],
                "bearing_deg": brg,
                "radius_m": r
            })

    df = pd.DataFrame(rows)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lon, df.lat),
        crs="EPSG:4326"
    )
    return gdf


In [4]:
# Radii of the pivot tracks
radii_m = [54.864, 109.728, 164.592, 219.456, 274.32, 329.184, 384.048]

# Tent points for field #4
field4_tents = gpd.read_file("/Users/taylorlee/Documents/TNT_Code/my_code/TNT/AgGPS/Data/TNTBees/BeeTents/#4/PointFeature.shp")
field4_tents = field4_tents.set_crs("EPSG:4326")

# Shapely polygons of the pivot tracks
pivot_tracks_polygon = geodesic_circle_polygons(49.910905, -111.76207, radii_m, n=360, start_bearing=0.0)

# Shapely points of the pivot tracks
pivot_tracks_points = geodesic_circle_points(49.910905, -111.76207, radii_m, n=200)

# Shapely polygon of the boundary
field4_boundary = gpd.read_file('/Users/taylorlee/Documents/TNT_Code/my_code/field_data/#4/boundaries/boundaries.shp')

In [ ]:
def get_lat_lon(point_geom):
    """
    Given a shapely Point geometry, returns (lat, lon) tuple.
    Assumes the geometry is in a geographic CRS (e.g., EPSG:4326).
    """
    if point_geom.is_empty or point_geom.geom_type != "Point":
        raise ValueError("Geometry must be a non-empty Point.")
    
    lon = point_geom.x  # stored first
    lat = point_geom.y  # stored second
    return lat, lon


def ll_at(lat1, lon1, bearing, distance):
    """
    Compute the destination point’s longitude and latitude given a start point, 
    an initial bearing, and a travel distance along a spherical Earth.

    Parameters
    ----------
    lon1 : float
        Starting longitude in decimal degrees.
    lat1 : float
        Starting latitude in decimal degrees.
    bearing : float
        Initial bearing (azimuth) in degrees clockwise from north.
    distance : float
        Distance to travel from the start point, in meters.

    Returns
    -------
    (lon2, lat2) : tuple of float
        Destination longitude and latitude in decimal degrees.

    Notes
    -----
    - Uses a spherical‐Earth model with radius R = 6,378,137 m (WGS-84).
    - Formulas are the standard “direct” geodesic solution:
      .. math::
         \\phi_2 = \\asin\\bigl(\\sin\\phi_1\\cos(d/R) + \\cos\\phi_1\\sin(d/R)\\cos\\theta\\bigr)

         \\lambda_2 = \\lambda_1 + \\operatorname{atan2}\\bigl(\\sin\\theta\\sin(d/R)\\cos\\phi_1,\\
         \\cos(d/R) - \\sin\\phi_1\\sin\\phi_2\\bigr)

    """
    R = 6378137 #Radius of the Earth in metres
    brng = math.radians(bearing)

    #lat2  52.20444 - the lat result I'm hoping for
    #lon2  0.36056 - the long result I'm hoping for.

    lat1 = math.radians(lat1) #Current lat point converted to radians
    lon1 = math.radians(lon1) #Current long point converted to radians

    lat2 = math.asin( math.sin(lat1)*math.cos(distance/R) +
         math.cos(lat1)*math.sin(distance/R)*math.cos(brng))

    lon2 = lon1 + math.atan2(math.sin(brng)*math.sin(distance/R)*math.cos(lat1),
                 math.cos(distance/R)-math.sin(lat1)*math.sin(lat2))

    return (math.degrees(lat2), math.degrees(lon2))


def verify_tent_position(
    tents: gpd.GeoDataFrame, 
    pivot_tracks: GeoDataFrame, 
    pivot_point: tuple[float, float], 
    min_dist: float, seed_angle: 
    float=0.0
    ) -> gpd.GeoDataFrame:
    """
        Verify that each tent is a minimum distance from the pivot tracks. Shift any 
        tents that are not a minimum distance from the pivot tracks. 

        Parameters
        ----------
        tents : GeoDataFrame
            Contains the tent coordinate points in the geometry column.
        pivot_tracks : GeoDataFrame
            Contains the pivot track coordinate points in the geometry column.
        pivot_point: Tuple
            Longitude and Latitude of the pivot point
        min_dist : float
            Minimum distance that the tents need to be from any pivot track
        seed_angle : float
            Compass angle the bays are seeded at.
        
        Returns
        -------
        gdf : A GeoDataFrame of the tents coordinates with any shifted tents

        Notes
        -----
        For any calculations involving geological data the coordinates need 
        to be passed in the order (lat, lon). The GeoDataFrame stores the coordinates
        in the order (lon, lat). This function has a test case for this to ensure the 
        correct coordinates are used. 
    """
    
    # Create a list of (lat, lon) tuples of the tent coordinates
    # The geometry column of the GeoDataFrame has the points in (lon, lat)
    # This puts the tent coordiantes into the correct order (lat, lon)
    tents = list(tents.geometry.apply(get_lat_lon))

    # Do the same for the pivot track points
    pivot_tracks = list(pivot_tracks.geometry.apply(get_lat_lon))

    #Create an empty list that will store tents that are not in the way of a pivot track. 
    new_tents = []

    # Convert pivot points to radians for haversine distance
    pivot_rad = np.radians(pivot_tracks)

    # Build the BallTree with haversine metric
    pivot_tree = BallTree(pivot_rad, metric='haversine')

    # The minimum distance that the tents need to be converted to radians for the
    # haversine metric 
    radius_rad = min_dist / 6371000.0

    # Loop through all the tents.
    for tent in tents:
        #Convert the tent coordinates to radians
        tent_rad = np.radians([tent])
        # Get the indices of any of the pivot track points in the radius of the tent
        indices = pivot_tree.query_radius(tent_rad, r=radius_rad)
        # Create a list of the pivot track points that are within the min radius of the tent
        nearby_points = [pivot_tracks[i] for i in indices[0]]

        # Check if there are pivot track points within the min distance of the tent
        if nearby_points:

            # Calculate how far to shift the tent using pythagorean theorem
            closest_point = min(nearby_points, key=lambda pt: geodesic(tent, pt).meters)

            p_dist = geodesic(pivot_point, closest_point).meters 

            t_dist = geodesic(pivot_point, tent).meters

            result = Geodesic.WGS84.Inverse(pivot_point[0], pivot_point[1], tent[0], tent[1])
            bearing = (result['azi1'] % 360)  # normalize to [0, 360)
            # Normalize so that seed_angle acts as the new 0° reference
            bearing = (bearing - seed_angle) % 360


            if p_dist <= t_dist: # tent is outside pivot circle
                if 270 <= bearing < 360 or 0 <= bearing < 90:
                    new_tent = ll_at(tent[0], tent[1], seed_angle, 1.0) # shift up

                if 90 <= bearing < 270:
                    new_tent = ll_at(tent[0], tent[1], seed_angle + 180.0, 1.0) # shift down
            
            else: # tent is inside pivot circle
                if 270 <= bearing < 360 or 0 <= bearing < 90:
                    new_tent = ll_at(tent[0], tent[1], seed_angle + 180.0, 1.0) #shift down
                if 90 <= bearing < 270:
                    new_tent = ll_at(tent[0], tent[1], seed_angle, 1.0) # shift up

            # Add the new tent location to the new tent list.
            new_tents.append(new_tent)

        # If there are no pivot track points within the min distance of the tent
        # add the tent to the new tents list and it will remain unchanged
        else:
            new_tents.append(tent) 


    #print(new_tents)
    # Convert new tent list to shapely Points (lon, lat order)
    geometry = [Point(lon, lat) for lat, lon in new_tents]

    # Return GeoDataFrame
    return gpd.GeoDataFrame(geometry=geometry, crs="EPSG:4326")

In [6]:
m = field4_tents.explore(color="blue", name="Tents")

m = pivot_tracks_polygon.explore(m=m, color="red", fill=False, name="Pivot Tracks")

#m = pivot_tracks_points.explore(m=m, color="red", name="Pivot Tracks")

#m = field4_boundary.explore(m=m, color="green")

m